In [ ]:
%load_ext autoreload
%autoreload 

In [ ]:
import pandas as pd
import numpy as np
from os import path, makedirs
from datetime import datetime
from functools import partial

# local imports
import sys
sys.path.append('../../../')
from pyanalib.split_df_helpers import *
from analysis_village.cc1pi.systematics.final_variable_configs import VariableConfig
from analysis_village.cc1pi.systematics.utils import *
from analysis_village.cc1pi.systematics.constants import *
from pyanalib.covariance import *
from analysis_village.cc1pi.DataFrameUtils.DFLoading import *
from analysis_village.cc1pi.Constants import CTE as CTE

from makedf.mcstat import get_MCstat_unc

from analysis_village.cc1pi.var_configs import *

# turn off PerformanceWarning 
# triggered by mismatched column levels
import warnings
warnings.filterwarnings("ignore", category=pd.errors.PerformanceWarning)

In [ ]:
save_result = True
save_fig = save_result

save_fig_base_dir = "/exp/sbnd/data/users/lpelegri/syst/"

today_str = datetime.now().strftime("%Y%m%d")
save_fig_dir = path.join(save_fig_base_dir, "systematics-other-{}".format(today_str))

if save_fig:
    if not path.exists(save_fig_dir):
        makedirs(save_fig_dir)
    print("saving plots in ", save_fig_dir)

# Load df

In [ ]:
pot_weight_col = ('slc', 'wgt', '', '', '', '')

#Load data
keys2load = ["cc1pi", "hdr", "histpotdf"] ## keys from the configuration file
data_df = load_df("/exp/sbnd/data/users/lpelegri/cafpyana_data/cc1pi_data_rollingdev_bnblight.df", keys2load, 100)
data_evt_df = data_df['cc1pi']
data_hdr_df = data_df['hdr']

# BNB data
# data_tot_pot = data_hdr_df['TOR875'].sum()
data_tot_pot = data_hdr_df['pot'].sum()
print("data_tot_pot: %.3e" %(data_tot_pot))
pot_str = f"{data_tot_pot} $\\times 10^{18}$"
data_evt_df[pot_weight_col] = np.ones(len(data_evt_df))
data_gates = data_hdr_df.nbnbinfo.sum()
print("data tot gates : %.3e" %(data_gates))




#Load CV dataframe
'''
CV_df = load_df("/scratch/7DayLifetime/lpelegrina/cc1pi_SystVarsCV.df", keys2load, 100)
CV_tot_pot =  CV_df['hdr']['pot'].sum()
CV_pot_scale = data_tot_pot / CV_tot_pot
CV_df['cc1pi'][pot_weight_col] = CV_pot_scale * np.ones(len(CV_df['cc1pi']))
'''

#Load systematics df
#syst_names = ["2xSCE", "PMTGainFluct","PMTHighNoise", "PMTLowEff"]

#syst_keys = ["SystVarsCV","wiremod_YZ","wiremod_XZ_thetaXW", "0xSCE", "CCalVar"]
#colors = ["black", "C0","C1","C2","C3"]
#labels    = ["CV", r"Wiremod Y-Z",r"Wiremod X$#theta_{xw}$", "0xSCE", "CCalVar"]

#syst_keys = ["SystVarsCV","wiremod_YZ","wiremod_XZ_thetaXW", "0xSCE", "ccalm"]
#colors = ["black", "C0","C1","C2","C3"]
#labels    = ["CV", r"Wiremod Y-Z",r"Wiremod X$\theta_{xw}$", "0xSCE","CCALM"]

syst_keys = ["SystVarsCV","wiremod_YZ","wiremod_XZ_thetaXW", "0xSCE","2xSCE","PMTGainFluct", "PMTHighNoise", "PMTLowEff"]
colors = ["black", "C0","C1","C2","C3","C4","C5","C6"]
labels    = ["CV", r"Wiremod Y-Z",r"Wiremod X$\theta_{xw}$", "0xSCE", "2xSCE","PMTGainFluct", "PMTHighNoise", "PMTLowEff"]

             
detvar_plotter = partial(
    variation_hists,
    var_colors=colors,
    var_labels=labels,
    approval="internal"
)

detvar_plotter_final_vars = partial(
    variation_hists_final_vars,
    var_colors=colors,
    var_labels=labels,
    approval="internal"
)

keys2load = ["cc1pi", "hdr", "histpotdf", "nudf"] ## keys from the configuration file
syst_dfs = {}
for key in syst_keys:
    syst_df = load_df(f"/exp/sbnd/data/users/lpelegri/cafpyana_data/cc1pi_{key}.df",keys2load,100)
        
    syst_tot_pot = syst_df['hdr']['pot'].sum()
    print("syst_pot: %.3e" %(syst_tot_pot))
    syst_pot_scale = data_tot_pot / syst_tot_pot
    syst_df['cc1pi'][pot_weight_col] = syst_pot_scale * np.ones(len(syst_df['cc1pi']))

    syst_dfs[key] = syst_df

In [ ]:
#CV_df['cc1pi'] = perform_truth_matching(CV_df['cc1pi'], CV_df['nudf'])
for key, df in syst_dfs.items():
    df['cc1pi'] = perform_truth_matching(df['cc1pi'], df['nudf'])

In [ ]:
#CV_evt_df = CV_df['cc1pi']
syst_evt_dfs = {}
for key, df in syst_dfs.items():
    syst_evt_dfs[key] = df['cc1pi']

In [ ]:
def build_event_masks(evt_df):

    masks = {}

    masks["obvious_cosmic"] = evt_df.slc.cut.obvious_cosmic
    masks["t0"]             = evt_df.slc.cut.t0
    masks["inside_FV"]      = evt_df.slc.cut.inside_FV
    masks["nu_score"]       = evt_df.slc.cut.nu_score
    masks["track"]          = evt_df.slc.cut.track
    masks["shower"]         = evt_df.slc.cut.shower
    masks["chi2"]           = evt_df.slc.cut.MIP_candidates
    masks["angle"]          = evt_df.slc.cut.angle
    masks["proton_BDT"]     = evt_df.slc.cut.proton_BDT
    masks["containment"]    = evt_df.slc.cut.containment
    masks["michel"]         = evt_df.slc.cut.michel
    masks["extra_pion"]     = evt_df.slc.cut.extra_pion
    masks["energy"]         = evt_df.slc.cut.energy

    return masks

In [ ]:
# --- CV ---
#CV_masks = build_event_masks(CV_evt_df)
syst_maks = {}
for key, df in syst_evt_dfs.items():
    masks = build_event_masks(df)
    syst_maks[key] = masks

# Check t0 distributions

In [ ]:
#CV_evt_df = CV_evt_df[CV_masks["obvious_cosmic"] & CV_masks["inside_FV"]]
for key in syst_evt_dfs:
    mask = syst_maks[key]["obvious_cosmic"] & syst_maks[key]["inside_FV"]
    syst_evt_dfs[key] = syst_evt_dfs[key][mask]

In [ ]:
evtdfs = [syst_evt_dfs[syst_key] for syst_key in syst_keys]
save_name = save_fig_dir + "/{}.png".format(config_bc_flash_score.file_name)
bc_flash_matcher_th = CTE.min_bc_score
n = detvar_plotter(evtdfs, 
                   config=config_bc_flash_score, 
                   vline=[bc_flash_matcher_th],
                   save_fig=save_fig, save_name=save_name,
                   data_tot_pot = data_tot_pot)

# Check nu score

In [ ]:
for key, df in syst_evt_dfs.items():
    mask = syst_maks[key]["t0"].reindex(df.index, fill_value=False)
    syst_evt_dfs[key] = df[mask]
    

In [ ]:
evtdfs = [syst_evt_dfs[syst_key] for syst_key in syst_evt_dfs]
save_name = save_fig_dir + "/{}.png".format(config_bc_flash_score.file_name)
bc_flash_matcher_th = CTE.min_bc_score
n = detvar_plotter(evtdfs, 
                   config=config_bc_flash_score, 
                   vline=[bc_flash_matcher_th],
                   save_fig=save_fig, save_name=save_name,
                   data_tot_pot = data_tot_pot)

In [ ]:
evtdfs = [syst_evt_dfs[syst_key] for syst_key in syst_evt_dfs]
save_name = save_fig_dir + "/{}.png".format(config_nu_score.file_name)
nu_score_th = CTE.min_nu_score
n = detvar_plotter(evtdfs, 
                   config=config_nu_score, 
                   vline=[nu_score_th],
                   save_fig=save_fig, save_name=save_name,
                   data_tot_pot = data_tot_pot)

# n tracks

In [ ]:
for key, df in syst_evt_dfs.items():
    mask = syst_maks[key]["nu_score"].reindex(df.index, fill_value=False)
    syst_evt_dfs[key] = df[mask]

In [ ]:
def plot_configs(configs, syst_evt_dfs):
    for config in configs:
    
        this_syst_evt_dfs = {}
        for key, df in syst_evt_dfs.items():
            this_syst_evt_dfs[key] = df[mask_dict[config.extra_mask](df)].copy()
        this_evtdfs = [this_syst_evt_dfs[syst_key] for syst_key in this_syst_evt_dfs]
        
        save_name = save_fig_dir + "/{}.png".format(config.file_name)
        n = detvar_plotter(this_evtdfs, 
                       config=config, 
                       vline=[100000000],
                       save_fig=save_fig, save_name=save_name,
                       data_tot_pot = data_tot_pot)

In [ ]:
configs = [config_pandora_primary_vtx_distance, config_analysis_primary_len, config_analysis_primary_track_score, config_n_prim_tracks]
plot_configs(configs, syst_evt_dfs)

# chi2 vars

# All

In [ ]:
for key, df in syst_evt_dfs.items():
    mask = syst_maks[key]["track"].reindex(df.index, fill_value=False)
    syst_evt_dfs[key] = df[mask]

In [ ]:
configs = [config_primary_track_chi2_mu, config_primary_track_chi2_p, config_primary_track_len]
plot_configs(configs, syst_evt_dfs)

# Candidates

In [ ]:
configs = [config_n_MIP_candidates, config_MIP_candidate_chi2_mu, config_MIP_candidate_chi2_p]
plot_configs(configs, syst_evt_dfs)

# After Cut

In [ ]:
for key, df in syst_evt_dfs.items():
    mask = syst_maks[key]["chi2"].reindex(df.index, fill_value=False)
    syst_evt_dfs[key] = df[mask]

# Candidates

In [ ]:
configs = [config_n_MIP_candidates, config_MIP_candidate_chi2_mu, config_MIP_candidate_chi2_p]
plot_configs(configs, syst_evt_dfs)

# Angle

In [ ]:
for key, df in syst_evt_dfs.items():
    mask = syst_maks[key]["shower"].reindex(df.index, fill_value=False) 
    syst_evt_dfs[key] = df[mask]

In [ ]:
configs = [config_angle_between_candidates]
plot_configs(configs, syst_evt_dfs)

# Proton BDT

In [ ]:
for key, df in syst_evt_dfs.items():
    mask = syst_maks[key]["angle"].reindex(df.index, fill_value=False) 
    syst_evt_dfs[key] = df[mask]

In [ ]:
configs = [config_MIP_candidate_chi2_mu, config_MIP_candidate_chi2_p, config_MIP_candidate_frac50, config_MIP_candidate_chi2_exp_pol, config_MIP_candidate_bdt_score_proton, config_n_MIP_candidates_proton]
plot_configs(configs, syst_evt_dfs)

# After cut

In [ ]:
for key, df in syst_evt_dfs.items():
    mask = syst_maks[key]["proton_BDT"].reindex(df.index, fill_value=False) 
    syst_evt_dfs[key] = df[mask]

In [ ]:
configs = [config_MIP_candidate_chi2_mu, config_MIP_candidate_chi2_p, config_MIP_candidate_frac50, config_MIP_candidate_chi2_exp_pol, config_MIP_candidate_bdt_score_proton, config_n_MIP_candidates_proton]
plot_configs(configs, syst_evt_dfs)

# Containment

In [ ]:
configs = [config_n_exiting_pfps, config_n_exiting_z_pfps]
plot_configs(configs, syst_evt_dfs)

# Michel

In [ ]:
for key, df in syst_evt_dfs.items():
    mask = syst_maks[key]["containment"].reindex(df.index, fill_value=False) 
    syst_evt_dfs[key] = df[mask]

In [ ]:
configs = [config_n_exiting_pfps, config_n_exiting_z_pfps]
plot_configs(configs, syst_evt_dfs)

In [ ]:
configs = [config_MIP_candidate_max_daughter_hits, config_contained_MIP_candidate_KE, config_contained_MIP_candidate_track_score, config_n_MIP_candidate_michel]
plot_configs(configs, syst_evt_dfs)

# Extra pion

In [ ]:
for key, df in syst_evt_dfs.items():
    mask = syst_maks[key]["michel"].reindex(df.index, fill_value=False) 
    syst_evt_dfs[key] = df[mask]

In [ ]:
configs = [config_n_extra_pions]
plot_configs(configs, syst_evt_dfs)

# Final Vars

In [ ]:
for key, df in syst_evt_dfs.items():
    mask = syst_maks[key]["extra_pion"].reindex(df.index, fill_value=False) &syst_maks[key]["energy"].reindex(df.index, fill_value=False)
    syst_evt_dfs[key] = df[mask]

In [ ]:
configs = [config_MIP_candidate_chi2_mu, config_MIP_candidate_chi2_p, config_MIP_candidate_chi2_exp_pol, config_MIP_candidate_max_daughter_hits, config_MIP_candidate_scatter_angle, config_MIP_candidate_bdt_score_muon_pion]
plot_configs(configs, syst_evt_dfs)

In [ ]:
configs = measure_var_vec + TKI_vec
plot_configs(configs, syst_evt_dfs)

# Get all the syst uncert

In [ ]:
var_config = VariableConfig.all_evts()

In [ ]:
slice_levels = ['__ntuple', 'entry', 'rec.slc..index']
evtdfs = [syst_evt_dfs[syst_key].groupby(level=slice_levels, sort=False).first() for syst_key in syst_evt_dfs.keys()]

var_name = var_config.var_evt_reco_col
bins = var_config.bins
pot_label = f"Candidate Slices (POT={pot_str})"
plot_labels = [var_config.var_labels[1], pot_label, ""]
approval = "internal"
save_name = save_fig_dir + "/{}.png".format(var_config.var_save_name)
n = detvar_plotter_final_vars(evtdfs, 
                    var_name=var_name,
                    bins=bins,
                    plot_labels=plot_labels,
                    approval=approval,
                    vline=[],
                    save_fig=save_fig, save_name=save_name)

In [ ]:
ret_dict = {}
for kidx, syst_key in enumerate(syst_dfs.keys()):
    if syst_key == "SystVarsCV":
        continue

    # take syst variation as a unisim uncertainty
    cv_events = n[0]
    univ_events = np.array([n[kidx]]) 
    ret = get_covariance_matrix(univ_events, cv_events)
    ret_dict[syst_key] = ret
    plot_univ_hists(univ_events, 
                    cv_events,
                    syst_key, 
                    var_config,
                    use_bin_width = False,
                    )

    matrix_type = "cov_frac"
    plot_labels = [var_config.var_labels[2], var_config.var_labels[1], ""]
    save_fig_name = "{}/{}-{}-{}.pdf".format(save_fig_dir, var_config.var_save_name, syst_key, matrix_type)
    title = "{} {}".format(syst_key, matrix_type)
    plot_heatmap(ret[matrix_type], 
                 bins=bins,
                 plot_labels=plot_labels,
                 save_fig=save_fig, 
                 save_name=save_fig_name)

    frac_unc = np.sqrt(np.diag(ret["cov_frac"]))
    plot_frac_unc([(frac_unc, syst_key)], var_config)
     

In [ ]:

# get total detector variation covariance matrix
for kidx, syst_key in enumerate(ret_dict.keys()):
    if kidx == 0:
        detvar_total_cov = ret_dict[syst_key]["cov_frac"]
    else:
        detvar_total_cov += ret_dict[syst_key]["cov_frac"]

frac_unc = np.sqrt(np.diag(detvar_total_cov))
plot_frac_unc([(frac_unc, "total")], var_config)

In [ ]:
file_dir = "/exp/sbnd/data/users/lpelegri/syst/frac_cov_matrices"
if 'syst_dict' not in locals():
    syst_dict = {}
os.makedirs(file_dir, exist_ok=True)  # create directory if needed

var_configs = [
    VariableConfig.all_evts(), 
    VariableConfig.muon_momentum(),
    VariableConfig.muon_direction(),
    VariableConfig.pion_momentum(),
    VariableConfig.pion_direction(),
    VariableConfig.angle_between_candidates(),
    VariableConfig.num_protons(),
    VariableConfig.delta_pt(),
    VariableConfig.delta_alpha_T(),
    VariableConfig.delta_phi_T()
]
syst_name = "detvar"

for var_config in var_configs:
    evtdfs = [syst_evt_dfs[syst_key].groupby(level=slice_levels, sort=False).first() for syst_key in syst_evt_dfs.keys()]
    var_name = var_config.var_evt_reco_col
    bins = var_config.bins
    pot_label = f"Candidate Slices (POT={pot_str})"
    plot_labels = [var_config.var_labels[1], pot_label, ""]
    approval = "internal"
    save_name = save_fig_dir + "/{}.png".format(var_config.var_save_name)
    n = detvar_plotter_final_vars(evtdfs, 
                        var_name=var_name,
                        bins=bins,
                        plot_labels=plot_labels,
                        approval=approval,
                        plot = True
                    )

    ret_dict = {}
    for kidx, syst_key in enumerate(syst_dfs.keys()):
        if syst_key == "SystVarsCV":
            continue
    
        # take syst variation as a unisim uncertainty
        cv_events = n[0]
        univ_events = np.array([n[kidx]]) 
        ret = get_covariance_matrix(univ_events, cv_events)
        ret_dict[syst_key] = ret
        '''
        plot_univ_hists(univ_events, 
                        cv_events,
                        syst_name, 
                        var_config,
                        use_bin_width = False,
                        )
    
        matrix_type = "cov_frac"
        plot_labels = [var_config.var_labels[2], var_config.var_labels[1], ""]
        save_fig_name = "{}/{}-{}-{}.pdf".format(save_fig_dir, var_config.var_save_name, syst_name, matrix_type)
        title = "{} {}".format(syst_key, matrix_type)

        plot_heatmap(ret[matrix_type], 
                     bins=bins,
                     plot_labels=plot_labels,
                     save_fig=save_fig, 
                     save_name=save_fig_name)
        '''
        
        frac_unc = np.sqrt(np.diag(ret["cov_frac"]))
        #plot_frac_unc([(frac_unc, syst_key)], var_config)

    # get total detector variation covariance matrix
    uncertanties_list = []
    for kidx, syst_key in enumerate(ret_dict.keys()):
        uncertanties_list.append((np.sqrt(np.diag(ret_dict[syst_key]["cov_frac"])), syst_key))
        if kidx == 0:
            detvar_total_cov = ret_dict[syst_key]["cov_frac"]
        else:
            detvar_total_cov += ret_dict[syst_key]["cov_frac"]
    
    uncertanties_list.append((frac_unc, "total"))
    frac_unc = np.sqrt(np.diag(detvar_total_cov))
    plot_frac_unc(uncertanties_list, var_config)

    syst_dict[var_config.var_save_name] = detvar_total_cov
    for syst_key in ret_dict.keys():
        syst_dict[var_config.var_save_name + "_" + syst_key] = ret_dict[syst_key]["cov_frac"]
    
# save syst_dict as an npz file in the directory where dfs were loaded from
if save_result:
    print("saving syst_dict as npz in %s" % (file_dir))
    np.savez(file_dir + "/detvar_syst_dict.npz", **syst_dict)